In [39]:
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm

adata = sc.read_h5ad('/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/southard_rpe1_tf_crispra_effect.h5ad')


adata_valid = adata[~adata.obs['sequence_driven']].copy()
weighted_X = []
guides = list(set(adata_valid.obs['target_gene']))
for guide in tqdm(guides):
    n_cells = adata_valid[adata_valid.obs['target_gene']==guide].obs['cell_count'].values
    effect = adata_valid[adata_valid.obs['target_gene']==guide].X
    weighted_effect = (n_cells@effect)/np.sum(n_cells)
    weighted_X.append(weighted_effect)
inferred_effectsize = pd.DataFrame(weighted_X, index=guides, columns=adata_valid.var.index.tolist())

100%|██████████| 1833/1833 [00:15<00:00, 118.59it/s]


In [43]:
centered_effectsize = inferred_effectsize - inferred_effectsize.mean(axis=0)

In [45]:
inferred_effectsize.to_csv('/nas/project1/PAGE_PJH/02.processed/southard_rpe1_filtered_noncentered_effectsize.csv.gz')
centered_effectsize.to_csv('/nas/project1/PAGE_PJH/02.processed/southard_rpe1_filtered_effectsize.csv.gz')